## Notebook summary

| Item | Details |
| --- | --- |
| Purpose | DenseNet-121 YOLO-ROI paired fine-tune from notebook 01 |
| Model | DenseNet-121 (from notebook 01 best checkpoint) |
| Input | Paired published crop + YOLO square ROI (alternates 50/50) |
| Training | Single-stage fine-tune, 5 epochs, AdamW + CosineAnnealingLR |
| Loss | Plain CrossEntropyLoss |
| Selection | QWK only |
| Outputs | best_model.pth, last_model.pth, history.csv, metadata.json |
| Status | Compact paired-view fine-tune (matches working baseline pattern) |


## Detailed config

### Identity

| Item | Value |
| --- | --- |
| Purpose | Fine-tune DenseNet-121 from notebook 01 on paired published+YOLO views |
| Base checkpoint | notebook 01 best_model.pth (original optimized) |
| Output dir | `/content/drive/MyDrive/Models/densenet121_yolo_roi/<TIMESTAMP>/` |

### Dataset

| Item | Value |
| --- | --- |
| Classes | 5 KL grades (0-4) |
| Published root | `/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224` |
| ROI root | `/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/densenet121_yolo_square_roi_trainvaltest_v2` |
| Input size | 384x384 |
| Augmentation | OpenCV CLAHE -> SquarePad -> PIL -> HFlip(p=0.5) -> Rotation(5) -> ColorJitter(0.08,0.08) -> Resize(384) -> RandomErasing(0.10) -> ImageNet norm |

### Training

| Item | Value |
| --- | --- |
| Epochs | 5 |
| LR | 1e-5 |
| Weight decay | 1e-3 |
| Batch size | 48 |
| Num workers | 2 |
| Scheduler | CosineAnnealingLR (stepped once per epoch) |
| Sampler | WeightedRandomSampler (inverse-class-frequency, power=1.0) |
| Loss | CrossEntropyLoss |
| Val views | Evaluated on both published and YOLO ROI views each epoch |

### Selection

| Item | Value |
| --- | --- |
| Robust selection | 0.5 * (published_selection + roi_selection) |
| Checkpoints | best_model.pth (max robust_selection), last_model.pth (every epoch) |


# DenseNet-121 YOLO-ROI Paired Fine-Tune

Fine-tune the notebook-01 DenseNet-121 checkpoint on paired published+YOLO views.
5 epochs, CosineAnnealingLR, WeightedRandomSampler, single-stage.
Matches the working paired-view baseline pattern.


## 0. Setup


In [1]:
!pip -q install 'timm>=1.0' 'h5py>=3.9'


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import json
import random
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, cohen_kappa_score, mean_absolute_error, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm


Mounted at /content/drive


## 1. Configuration


In [3]:
# ─── Paths ───────────────────────────────────────────────────────────────────
PUBLISHED_ROOT = Path(
    '/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/'
    'extracted/KneeXrayData/ClsKLData/kneeKL224'
)
ROI_ROOT = Path(
    '/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/'
    'derived/densenet121_yolo_square_roi_trainvaltest_v2'
)
# Base checkpoint = notebook 01's best_model.pth (auto-discovers the latest
# completed run under the optimized_original directory; fails fast if none exist).
BASE_CHECKPOINT_ROOT = Path('/content/drive/MyDrive/Models/densenet121_optimized_original')
candidates = sorted(
    BASE_CHECKPOINT_ROOT.glob('*/best_model.pth'),
    key=lambda p: p.stat().st_mtime, reverse=True
)
if not candidates:
    raise FileNotFoundError(
        f'No best_model.pth under {BASE_CHECKPOINT_ROOT}. '
        'Run notebook 01 first to produce a base checkpoint.'
    )
BASE_CHECKPOINT = candidates[0]
print(f'Base checkpoint (latest by mtime): {BASE_CHECKPOINT}')

# ─── Training ─────────────────────────────────────────────────────────────────
SEED = 42
INPUT_SIZE = 384
NUM_WORKERS = 2
IS_A100 = torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0)
BATCH_SIZE = 32 if IS_A100 else 16
NUM_WORKERS = 8 if IS_A100 else 2
PERSISTENT_WORKERS = False   # don't reserve GPU memory via idle workers
EPOCHS = 5
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-3
ALTERNATE_VIEW_PROBABILITY = 0.50  # 50 % chance to use YOLO ROI instead of published crop

# ─── Derived ────────────────────────────────────────────────────────────────
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_%f_UTC')
RUN_DIR = Path('/content/drive/MyDrive/Models/densenet121_yolo_roi') / RUN_TIMESTAMP

for p in (PUBLISHED_ROOT, ROI_ROOT, BASE_CHECKPOINT):
    if not p.exists():
        raise FileNotFoundError(p)
RUN_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print(f'Base checkpoint: {BASE_CHECKPOINT}')
print(f'Epochs: {EPOCHS}  LR: {LEARNING_RATE}  Batch size: {BATCH_SIZE}  Workers: {NUM_WORKERS}')
print(f'Scheduler: CosineAnnealingLR (stepped once per epoch)')
print(f'Alternate view probability: {ALTERNATE_VIEW_PROBABILITY}')


Base checkpoint (latest by mtime): /content/drive/MyDrive/Models/densenet121_optimized_original/2026-08-20_15-15-22_677034_UTC/best_model.pth
Device: cuda
Base checkpoint: /content/drive/MyDrive/Models/densenet121_optimized_original/2026-08-20_15-15-22_677034_UTC/best_model.pth
Epochs: 5  LR: 1e-05  Batch size: 16  Workers: 2
Scheduler: CosineAnnealingLR (stepped once per epoch)
Alternate view probability: 0.5


## 2. Build paired published / YOLO records


In [4]:
rows = []
for split in ('train', 'val'):
    for grade in range(5):
        for pub in sorted((PUBLISHED_ROOT / split / str(grade)).glob('*.png')):
            roi = ROI_ROOT / split / str(grade) / pub.name
            if not roi.is_file():
                raise FileNotFoundError(f'Missing paired ROI: {roi}')
            rows.append({'split': split, 'grade': grade,
                         'published_path': str(pub), 'roi_path': str(roi)})

frame = pd.DataFrame(rows)
print(frame.groupby(['split', 'grade']).size().unstack(fill_value=0))

train_frame = frame[frame.split == 'train'].reset_index(drop=True)
val_frame   = frame[frame.split == 'val'  ].reset_index(drop=True)

counts = np.bincount(train_frame.grade.to_numpy(), minlength=5)
weights = (1.0 / counts)[train_frame.grade.to_numpy()]
sampler = WeightedRandomSampler(
    torch.as_tensor(weights, dtype=torch.double), len(weights), replacement=True
)
print(f'\nClass counts: {dict(enumerate(counts))}')


grade     0     1     2    3    4
split                            
train  2286  1046  1516  757  173
val     328   153   212  106   27

Class counts: {0: np.int64(2286), 1: np.int64(1046), 2: np.int64(1516), 3: np.int64(757), 4: np.int64(173)}


## 3. Preprocessing, Dataset & Model


In [5]:
class OpenCVCLAHE:
    def __call__(self, image_rgb):
        lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l = cv2.createCLAHE(clipLimit=1.25, tileGridSize=(8, 8)).apply(l)
        return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)

class SquarePad:
    def __call__(self, image_rgb):
        h, w = image_rgb.shape[:2]
        side = max(h, w)
        top = (side - h) // 2
        left = (side - w) // 2
        return cv2.copyMakeBorder(
            image_rgb, top, side - h - top, left, side - w - left,
            cv2.BORDER_CONSTANT, value=(0, 0, 0))

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.50),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0),
    normalize,
])

val_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    normalize,
])

class PairedDataset(Dataset):
    def __init__(self, data, transform, alternate_probability):
        self.data = data.reset_index(drop=True)
        self.transform = transform
        self.alternate_probability = alternate_probability
        self.labels = self.data.grade.astype(int).tolist()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        use_roi = (self.alternate_probability > 0 and
                   random.random() < self.alternate_probability)
        img = cv2.imread(row.roi_path if use_roi else row.published_path)
        if img is None:
            raise IOError(f'Cannot read image at index {index}')
        return self.transform(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)), int(row.grade)


class DenseNet121Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            'densenet121', pretrained=False, num_classes=5, drop_rate=0.20)

    def forward(self, images):
        return self.backbone(images)


model = DenseNet121Model().to(DEVICE)
checkpoint = torch.load(BASE_CHECKPOINT, map_location='cpu', weights_only=False)
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)
print(f'Loaded base checkpoint: {BASE_CHECKPOINT}')
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total:,}  Trainable: {trainable:,} (all layers unfrozen)')

train_loader = DataLoader(
    PairedDataset(train_frame, train_transform, ALTERNATE_VIEW_PROBABILITY),
    batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_pub_loader = DataLoader(
    PairedDataset(val_frame, val_transform, 0.0),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_roi_loader = DataLoader(
    PairedDataset(val_frame, val_transform, 1.0),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(f'Train batches: {len(train_loader)}')
print(f'Val batches (published): {len(val_pub_loader)}')
print(f'Val batches (ROI):       {len(val_roi_loader)}')


Loaded base checkpoint: /content/drive/MyDrive/Models/densenet121_optimized_original/2026-08-20_15-15-22_677034_UTC/best_model.pth
Total parameters: 6,958,981  Trainable: 6,958,981 (all layers unfrozen)
Train batches: 362
Val batches (published): 52
Val batches (ROI):       52


## 4. Training Loop


In [6]:
# ─── Plain CrossEntropyLoss — no mixup, no ordinal tricks ─────────────
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn.functional as F

def make_optimizer(model):
    return torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


def evaluate_published(loader):
    model.eval()
    all_labels, all_preds, all_probas = [], [], []
    with torch.inference_mode():
        for images, labels in tqdm(loader, desc='[Pub]'):
            images = images.to(DEVICE, non_blocking=True)
            logits = model(images).float()
            probas = F.softmax(logits, dim=1).cpu().numpy()
            all_labels.extend(labels.numpy())
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_probas.extend(probas)
    y_true = np.asarray(all_labels).astype(int)
    y_pred = np.asarray(all_preds).astype(int)
    y_proba = np.asarray(all_probas)
    y_onehot = np.eye(5)[y_true]
    qwk = float(cohen_kappa_score(y_true, y_pred, weights='quadratic'))
    _, _, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    macro_f1 = float(f1)
    macro_ap = float(average_precision_score(y_onehot, y_proba, average='macro'))
    return {
        "accuracy": float(np.mean(y_true == y_pred)),
        "qwk": qwk,
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "off1_acc": float(np.mean(np.abs(y_true - y_pred) <= 1)),
        "macro_f1": macro_f1,
        "macro_ap": macro_ap,
    }


def evaluate_roi(loader):
    model.eval()
    all_labels, all_preds, all_probas = [], [], []
    with torch.inference_mode():
        for images, labels in tqdm(loader, desc='[ROI]'):
            images = images.to(DEVICE, non_blocking=True)
            logits = model(images).float()
            probas = F.softmax(logits, dim=1).cpu().numpy()
            all_labels.extend(labels.numpy())
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_probas.extend(probas)
    y_true = np.asarray(all_labels).astype(int)
    y_pred = np.asarray(all_preds).astype(int)
    y_proba = np.asarray(all_probas)
    y_onehot = np.eye(5)[y_true]
    qwk = float(cohen_kappa_score(y_true, y_pred, weights='quadratic'))
    _, _, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    macro_f1 = float(f1)
    macro_ap = float(average_precision_score(y_onehot, y_proba, average='macro'))
    return {
        "accuracy": float(np.mean(y_true == y_pred)),
        "qwk": qwk,
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "off1_acc": float(np.mean(np.abs(y_true - y_pred) <= 1)),
        "macro_f1": macro_f1,
        "macro_ap": macro_ap,
    }


In [7]:
# ─── Training loop: fine-tune from notebook 01 checkpoint, alternating views ──
LR_STAGE_HEAD = LEARNING_RATE
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_STAGE_HEAD, weight_decay=WEIGHT_DECAY)

# Cosine schedule across the full EPOCHS budget
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')

criterion = nn.CrossEntropyLoss()

best_score = -float('inf')
history = []

last_checkpoint_path = RUN_DIR / 'last_model.pth'
best_checkpoint_path = RUN_DIR / 'best_model.pth'

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for images, labels in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item() * labels.size(0)
        epoch_correct += (logits.argmax(dim=1) == labels).sum().item()
        epoch_total += labels.size(0)
        progress.set_postfix(loss=f'{epoch_loss / epoch_total:.4f}', acc=f'{epoch_correct / epoch_total:.4f}')

    scheduler.step()
    train_loss = epoch_loss / max(epoch_total, 1)
    train_acc = epoch_correct / max(epoch_total, 1)

    pub_metrics = evaluate_published(val_pub_loader)
    roi_metrics = evaluate_roi(val_roi_loader)
    # Robust selection = mean of the two QWKs (eval-views must agree on improvements)
    robust_selection = 0.5 * (pub_metrics['qwk'] + roi_metrics['qwk'])

    history.append({
        'epoch': epoch,
        'lr': optimizer.param_groups[0]['lr'],
        'train_loss': train_loss,
        'train_acc': train_acc,
        'pub_qwk': pub_metrics['qwk'],
        'pub_acc': pub_metrics['accuracy'],
        'pub_macro_f1': pub_metrics['macro_f1'],
        'roi_qwk': roi_metrics['qwk'],
        'roi_acc': roi_metrics['accuracy'],
        'roi_macro_f1': roi_metrics['macro_f1'],
        'robust_selection': robust_selection,
    })

    print(
        f'Epoch {epoch}: loss={train_loss:.4f} acc={train_acc:.4f} | '
        f'pub_qwk={pub_metrics["qwk"]:.4f} roi_qwk={roi_metrics["qwk"]:.4f} | '
        f'robust={robust_selection:.4f}'
    )

    payload = {
        'model_state_dict': model.state_dict(),
        'selection': robust_selection,
        'pub_metrics': pub_metrics,
        'roi_metrics': roi_metrics,
    }
    torch.save(payload, last_checkpoint_path)

    if robust_selection > best_score:
        best_score = robust_selection
        torch.save(payload, best_checkpoint_path)
        print(f'  -> New best! Robust={best_score:.4f}')


Epoch 1/5:   0%|          | 0/362 [00:00<?, ?it/s]

[Pub]:   0%|          | 0/52 [00:00<?, ?it/s]

NameError: name 'mean_absolute_error' is not defined

In [ ]:
# ─── Final evaluation: load best checkpoint and report comprehensive metrics ──
best_ckpt = torch.load(best_checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(best_ckpt["model_state_dict"])
print(f"Loaded best checkpoint (selection={best_ckpt['selection']:.4f})\n")

pub_m = evaluate_published(val_pub_loader)
roi_m = evaluate_roi(val_roi_loader)

# Print clean summary table
print("=" * 70)
print(f"{'':30s}  {'Published':>12s}  {'YOLO-ROI':>12s}  {'Robust':>12s}")
print("-" * 70)
print(f"{'Accuracy':30s}  {pub_m['accuracy']:12.4f}  {roi_m['accuracy']:12.4f}  {(pub_m['accuracy']+roi_m['accuracy'])/2:12.4f}")
print(f"{'QWK':30s}  {pub_m['qwk']:12.4f}  {roi_m['qwk']:12.4f}  {(pub_m['qwk']+roi_m['qwk'])/2:12.4f}")
print(f"{'MAE':30s}  {pub_m['mae']:12.4f}  {roi_m['mae']:12.4f}  {(pub_m['mae']+roi_m['mae'])/2:12.4f}")
print(f"{'Off-by-1 Accuracy':30s}  {pub_m['off1_acc']:12.4f}  {roi_m['off1_acc']:12.4f}  {(pub_m['off1_acc']+roi_m['off1_acc'])/2:12.4f}")
print(f"{'Macro F1':30s}  {pub_m['macro_f1']:12.4f}  {roi_m['macro_f1']:12.4f}  {(pub_m['macro_f1']+roi_m['macro_f1'])/2:12.4f}")
print(f"{'Macro AP':30s}  {pub_m['macro_ap']:12.4f}  {roi_m['macro_ap']:12.4f}  {(pub_m['macro_ap']+roi_m['macro_ap'])/2:12.4f}")
print("=" * 70)
print(f"{'Best Robust Selection':30s}  {'':>12s}  {'':>12s}  {best_score:12.4f}")
print("=" * 70)

final_metrics = {
    "best_robust_selection": float(best_score),
    "published": pub_m,
    "yolo_roi": roi_m,
}
with open(RUN_DIR / "final_metrics.json", "w") as f:
    json.dump(final_metrics, f, indent=2)
print(f'\nSaved: {RUN_DIR / "final_metrics.json"}')


## 5. Save History & Metadata


In [ ]:
pd.DataFrame(history).to_csv(RUN_DIR / 'history.csv', index=False)

metadata = {
    'architecture': 'densenet121_yolo_roi',
    'loss': 'cross_entropy',
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'persistent_workers': PERSISTENT_WORKERS,
    'num_workers': NUM_WORKERS,
    'input_size': INPUT_SIZE,
    'alternate_view_probability': ALTERNATE_VIEW_PROBABILITY,
    'scheduler': 'cosine_annealing',
    'best_robust_selection': best_score,
    'base_checkpoint': str(BASE_CHECKPOINT),
    'published_root': str(PUBLISHED_ROOT),
    'roi_root': str(ROI_ROOT),
    'train_samples': len(train_frame),
    'val_samples': len(val_frame),
}
with open(RUN_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved:')
print(f'  {RUN_DIR / "best_model.pth"}')
print(f'  {RUN_DIR / "last_model.pth"}')
print(f'  {RUN_DIR / "history.csv"}')
print(f'  {RUN_DIR / "metadata.json"}')
print(f'\nBest robust_selection: {best_score:.4f}')
print(f'Next: run evaluation notebook')


NameError: name 'best_score' is not defined